# Practical PyTorch II · Part 4 — Companion Notebook

This goes with **"Transfer Learning: Standing on a Pretrained Model's Shoulders"**. Run the cells top to bottom (Shift+Enter).

A GPU helps but isn't required — this is small enough to finish on CPU in a few minutes. For the GPU: **Runtime → Change runtime type → Hardware accelerator → GPU**, then run the first cell again.

## 1. Setup

Colab already has PyTorch and torchvision. We just confirm what we've got and pick a device.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

print("PyTorch:", torch.__version__)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

## 2. A tiny dataset: ants vs. bees

We'll use the classic *hymenoptera* dataset — about 120 training images per class, two classes. It downloads in seconds. The point isn't the bugs; it's that we'll build a working classifier from this little data, which is only possible because we're standing on a pretrained model.

In [ ]:
import os

if not os.path.exists("hymenoptera_data"):
    !wget -q https://download.pytorch.org/tutorial/hymenoptera_data.zip
    !unzip -q hymenoptera_data.zip

print("train classes:", os.listdir("hymenoptera_data/train"))
print("  ants:", len(os.listdir("hymenoptera_data/train/ants")), "images")
print("  bees:", len(os.listdir("hymenoptera_data/train/bees")), "images")

## 3. Preprocess inputs the way the model expects

Pretrained weights were trained on images resized, cropped, and normalized a specific way. torchvision hands us the exact transforms via the weights object, so we don't have to guess the magic numbers. We add a little random flip/crop on the training set for variety.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

weights = ResNet18_Weights.DEFAULT

# The exact preprocessing the pretrained backbone expects.
eval_tf = weights.transforms()
print(eval_tf)

# A touch of augmentation for training, then the same normalize as eval.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

train_ds = datasets.ImageFolder("hymenoptera_data/train", transform=train_tf)
val_ds = datasets.ImageFolder("hymenoptera_data/val", transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=16)

num_classes = len(train_ds.classes)
print("classes:", train_ds.classes, "-> num_classes =", num_classes)

## 4. Load the model, freeze the backbone, swap the head

The whole transfer-learning move in three steps: load a pretrained model, freeze every parameter, then replace the final layer with a fresh one sized for **our** classes. Order matters — freeze first, swap second — so the new head comes back trainable.

In [ ]:
model = resnet18(weights=weights)

# Freeze the entire backbone.
for p in model.parameters():
    p.requires_grad = False

# Swap in a fresh head for our number of classes (trainable by default).
model.fc = nn.Linear(model.fc.in_features, num_classes)

model = model.to(device)

# Confirm: only the head's parameters are trainable.
trainable = [name for name, p in model.named_parameters() if p.requires_grad]
print("trainable params:", trainable)

## 5. Train just the head

This is the loop from Part 2, with one change: the optimizer only gets the **trainable** parameters. The backbone still runs forward each batch (it produces features), but because it's frozen, `step()` only moves the head.

In [ ]:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3,
)
loss_fn = nn.CrossEntropyLoss()

num_epochs = 3
for epoch in range(num_epochs):
    model.train()
    running = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        running += loss.item() * images.size(0)
    print(f"epoch {epoch + 1}: train loss = {running / len(train_ds):.4f}")

## 6. How did it do?

We switch to `model.eval()` (ResNet has batch-norm layers that behave differently at inference), turn off gradient tracking, and count correct predictions on the held-out validation set. From ~240 training images and a few minutes, you should land well into the 90s — that's the pretrained backbone doing the heavy lifting.

In [ ]:
model.eval()
correct = total = 0
with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        preds = model(images).argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

print(f"validation accuracy: {correct / total:.1%}  ({correct}/{total})")

## 7. Optional: full fine-tuning

Feature extraction (frozen backbone) is the right default. If your images were unusual and you had more data, you'd *unfreeze* the backbone and train everything — at a **much smaller learning rate** so you don't scribble over the pretrained features. Here's the shape of it (skip it if you're happy with the result above):

In [ ]:
# Unfreeze the whole model...
for p in model.parameters():
    p.requires_grad = True

# ...and fine-tune everything at a 10x-smaller learning rate.
optimizer = optim.Adam(model.parameters(), lr=1e-4)

model.train()
for images, labels in train_loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    loss = loss_fn(model(images), labels)
    loss.backward()
    optimizer.step()

print("one epoch of full fine-tuning done — note lr=1e-4, not 1e-3")

That's transfer learning end to end: reuse a pretrained backbone, swap the head for your classes, freeze (or don't), and train with the loop you already know. Next up in the series — handing the boilerplate to Hugging Face's `Trainer`.